# Build event config LaTeX table

Reads `config_romy_<event>.yml` files for a list of events and creates a LaTeX table.

Columns:
- Origin time (UTC, date and time split over two lines)
- Magnitude
- Data start time (UTC, date and time split over two lines)
- Data end time (UTC, date and time split over two lines)
- Cut outs (components among Z, N, E that have `time_cuts`)

LaTeX note:
- The table uses `\\shortstack{...}` for line breaks inside cells.
- `\\shortstack` is built into LaTeX (no extra package required).
- The table uses **booktabs** rules (`\\toprule`, `\\midrule`, `\\bottomrule`), so add `\\usepackage{booktabs}`.


## Notebook Overview
This notebook compiles event configuration metadata into a concise summary table (including LaTeX-ready output).

**Main workflow**
- Load event list and per-event config files.
- Extract selected configuration fields.
- Build a unified tabular overview.
- Format table for documentation/publication use.
- Export or print the final table representation.

## Imports

In [96]:
import os
import re
import yaml
import pandas as pd


## Configuration

In [97]:
# Option A: use list from YAML file
events_list_file = "./events_list.yml"

# Option B: explicit list (set to None to use file)
events_override = None
# events_override = ["20240314", "20240322", "20240414"]

events_root = "./events"

# Output .tex path (optional)
save_tex = False
tex_out_path = "./output/tables/events_config_table.tex"


## Load event list

In [98]:
def load_events(events_file=None, events_override=None):
    if events_override:
        return [str(e) for e in events_override]
    if not events_file:
        return []
    with open(events_file, "r") as f:
        cfg = yaml.safe_load(f)
    return [str(e) for e in cfg.get("events", [])]


events = load_events(events_list_file, events_override)
print(f"Loaded {len(events)} events")
events[:5]


Loaded 18 events


['20190514', '20230908', '20230912', '20231202', '20240314']

## Parse config files

In [99]:
def load_yaml_with_fallback(path):
    with open(path, "r") as f:
        txt = f.read()
    # Handle files with !!python/tuple by progressively relaxing loader
    for loader in (yaml.safe_load, yaml.full_load, yaml.unsafe_load):
        try:
            return loader(txt)
        except Exception:
            continue
    raise ValueError(f"Could not parse YAML: {path}")


def split_utc_lines(ts):
    ts = str(ts) if ts is not None else ""
    # Supports 'YYYY-mm-dd HH:MM:SS' and ISO with T/Z
    s = ts.replace("T", " ").replace("Z", "")
    parts = s.split()
    if len(parts) >= 2:
        return parts[0], parts[1]
    if len(parts) == 1:
        return parts[0], ""
    return "", ""


def latex_time_cell(ts):
    d, t = split_utc_lines(ts)
    # shortstack keeps compact multi-line content in a table cell
    # add a little breathing room between date and time
    return rf"\shortstack{{{d}\\[0.4ex]{t}}}"


def cut_components_from_time_cuts(time_cuts):
    if not isinstance(time_cuts, dict) or len(time_cuts) == 0:
        return "-"

    found = set()
    for key in time_cuts.keys():
        k = str(key).upper()
        # prefer trailing component marker (e.g. HZ, JN, BHE)
        m = re.search(r"([ZNE])$", k)
        if m:
            found.add(m.group(1))
            continue
        # fallback: any contained component letter
        for c in "ZNE":
            if c in k:
                found.add(c)

    ordered = [c for c in ["Z", "N", "E"] if c in found]
    return ",".join(ordered) if ordered else "-"


rows = []
missing = []

for ev in events:
    cfg_path = os.path.join(events_root, f"event_{ev}", f"config_romy_{ev}.yml")
    if not os.path.exists(cfg_path):
        missing.append(cfg_path)
        continue

    cfg = load_yaml_with_fallback(cfg_path)

    rows.append({
        "event": ev,
        "origin_time": cfg.get("origin_time", ""),
        "magnitude": cfg.get("magnitude", ""),
        "tbeg": cfg.get("tbeg", ""),
        "tend": cfg.get("tend", ""),
        "cut_outs": cut_components_from_time_cuts(cfg.get("time_cuts", {})),
    })

if missing:
    print("Missing config files:")
    for m in missing:
        print(" -", m)

print(f"Parsed {len(rows)} event configs")
df = pd.DataFrame(rows)
df.head()


Parsed 18 event configs


,event,origin_time,magnitude,tbeg,tend,cut_outs
0,20190514,2019-05-14 12:58:28,7.5,2019-05-14 13:15:00,2019-05-14 14:30:00,-
1,20230908,2023-09-08 22:11:00,6.8,2023-09-08 22:14:30,2023-09-08 22:43:00,-
2,20230912,2023-09-12 11:03:17,6.8,2023-09-12 11:12:00,2023-09-12 12:15:00,-
3,20231202,2023-12-02 14:37:04,7.6,2023-12-02 14:46:00,2023-12-02 15:55:00,"N,E"
4,20240314,2024-03-14 21:10:24,6.6,2024-03-14 21:15:00,2024-03-14 22:00:00,-


## Build LaTeX table

In [100]:
def build_latex_table(rows, caption="", label="tab:data_intervals"):
    header = [
        r"\begin{table}[h!]",
        r"\centering",
        r"\renewcommand{\arraystretch}{1.12}",
        f"\\caption{{{caption}}}",
        f"\\label{{{label}}}",
        r"\begin{tabular}{ccccc}",
        r"\toprule",
        r"\shortstack{Origin time\\(UTC)} & \shortstack{Catalog\\magnitude} & \shortstack{Data start\\(UTC)} & \shortstack{Data end\\(UTC)} & \shortstack{Data gaps\\(Z,N,E)} \\",

        r"\midrule",
    ]

    def latex_center_cell(text):
        return rf"\shortstack[c]{{{text}\\[0.8ex]}}"

    body = []
    for r in rows:
        c_origin = latex_time_cell(r["origin_time"])
        c_mag = latex_center_cell(f"{float(r['magnitude']):.1f}") if r["magnitude"] != "" else latex_center_cell("")
        c_tbeg = latex_time_cell(r["tbeg"])
        c_tend = latex_time_cell(r["tend"])
        c_cut = latex_center_cell(r["cut_outs"])
        body.append(f"{c_origin} & {c_mag} & {c_tbeg} & {c_tend} & {c_cut} \\\\")

    footer = [
        "",
        r"\bottomrule",
        r"\end{tabular}",
        r"\end{table}",
    ]

    return "\n".join(header + body + footer)


latex_table = build_latex_table(rows)
print(latex_table)


\begin{table}[h!]
\centering
\renewcommand{\arraystretch}{1.12}
\caption{}
\label{tab:data_intervals}
\begin{tabular}{ccccc}
\toprule
\shortstack{Origin time\\(UTC)} & \shortstack{Catalog\\magnitude} & \shortstack{Data start\\(UTC)} & \shortstack{Data end\\(UTC)} & \shortstack{Data gaps\\(Z,N,E)} \\
\midrule
\shortstack{2019-05-14\\[0.4ex]12:58:28} & \shortstack[c]{7.5\\[0.8ex]} & \shortstack{2019-05-14\\[0.4ex]13:15:00} & \shortstack{2019-05-14\\[0.4ex]14:30:00} & \shortstack[c]{-\\[0.8ex]} \\
\shortstack{2023-09-08\\[0.4ex]22:11:00} & \shortstack[c]{6.8\\[0.8ex]} & \shortstack{2023-09-08\\[0.4ex]22:14:30} & \shortstack{2023-09-08\\[0.4ex]22:43:00} & \shortstack[c]{-\\[0.8ex]} \\
\shortstack{2023-09-12\\[0.4ex]11:03:17} & \shortstack[c]{6.8\\[0.8ex]} & \shortstack{2023-09-12\\[0.4ex]11:12:00} & \shortstack{2023-09-12\\[0.4ex]12:15:00} & \shortstack[c]{-\\[0.8ex]} \\
\shortstack{2023-12-02\\[0.4ex]14:37:04} & \shortstack[c]{7.6\\[0.8ex]} & \shortstack{2023-12-02\\[0.4ex]14:46:00} & \sh

## Optional: save LaTeX to file

In [ ]:
# if save_tex:
#     os.makedirs(os.path.dirname(tex_out_path), exist_ok=True)
#     with open(tex_out_path, "w") as f:
#         f.write(latex_table)
#     print(f"Saved: {tex_out_path}")
